# API requests, limits and the frozen demonstration model

Upload the complete project ZIP if using Colab. Use Python 3.12 for the verified dependency lock. Run all cells. The original dashboard model is a frozen historical demonstration, not the preferred research method.

In [1]:
import sys, os, json, subprocess, tempfile, zipfile
from pathlib import Path
from IPython.display import display, Markdown
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path('/content/delaylens')] if (p/'experiments/REVIEW_PROTOCOL.md').exists()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploads = files.upload()
    archive = next(name for name in uploads if name.endswith('.zip'))
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            parts = Path(name).parts
            if not parts or parts[0] != 'delaylens' or '..' in parts or Path(name).is_absolute():
                raise ValueError('Unexpected archive member')
        z.extractall('/content')
    ROOT = Path('/content/delaylens')
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted DelayLens project.')
os.chdir(ROOT)
PYTHON = sys.executable
if IN_COLAB:
    env = Path('/content/delaylens-review-env')
    if not (env/'bin/python').exists():
        subprocess.run([sys.executable,'-m','venv',str(env)],check=True)
    PYTHON = str(env/'bin/python')
    subprocess.run([PYTHON,'-m','pip','install','-r','requirements-lock.txt'],cwd=ROOT,check=True)
def run_python(source):
    result = subprocess.run([PYTHON,'-c',source],cwd=ROOT,text=True,capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        result.check_returncode()
print('Ready. This notebook reads real saved evidence; it does not refit candidates.')


Ready. This notebook reads real saved evidence; it does not refit candidates.


## Valid request
The API retains the original fitted artifact so its existing probability contract stays explicit. The new evaluation does not silently replace it.

In [2]:
run_python("import json\nfrom fastapi.testclient import TestClient\nfrom app.main import create_app\nwith TestClient(create_app()) as c:\n    order=json.load(open('examples/single_order.json'))\n    r=c.post('/predict',json=order)\n    assert r.status_code==200\n    print(r.json())\n    order['order_delivered_customer_date']='2018-07-31'\n    bad=c.post('/predict',json=order)\n    assert bad.status_code==422\n    print('Future field rejected:',bad.status_code)\n    large=c.post('/predict',content=b'x'*1000001)\n    assert large.status_code==413\n    print('Oversized body rejected:',large.status_code)")

{'order_id': 'DEMO-1001', 'delay_probability': 0.271671327417174, 'risk_band': 'high', 'model_version': 'delaylens-1.0-a967c698-94eced67', 'release_status': 'research_only', 'warnings': []}
Future field rejected: 422
Oversized body rejected: 413



## Actual verification evidence
The saved test report includes a separate live uvicorn request and a chunked oversized request over loopback. Test counts come from pytest XML.

In [3]:
print((ROOT/'reports/review/verification.json').read_text())

{
  "tests": 41,
  "failures": 0,
  "errors": 0,
  "skipped": 0,
  "live_http_post_preserved": true,
  "live_chunked_oversize_413": true,
  "passed": 41,
  "packages": {
    "pytest": "8.3.5",
    "fastapi": "0.115.12",
    "starlette": "0.46.2",
    "httpx": "0.28.1",
    "numpy": "2.3.5",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.8.0",
    "joblib": "1.5.3"
  }
}

